# Chapter 13 &mdash; A DTM for $w\#w$, an NDTM for $ww$

**Concept 9 of the Chapter 13 decomposition:** *A DTM for $w\#w$, an NDTM for $ww$, and Why Nondeterminism Adds No Power*

The separator makes $w\#w$ deterministic; $ww$ needs a guessed midpoint &mdash; but a DTM can find it too.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-DTM-For-wsw-And-NDTM-For-ww/Concept-DTM-For-wsw-And-NDTM-For-ww.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$w\#w$ and $ww$ differ by one symbol, and that symbol changes the design completely.

**$w\#w$ is deterministic.** The `#` tells you where the halves meet: cross off the
leftmost unmarked symbol on the left, walk past the `#`, check and cross off the
matching one on the right, return, repeat.

**$ww$ has no separator.** The natural machine **guesses** the midpoint &mdash; a
nondeterministic TM, as in Chapter 12's palindrome PDA.

But unlike PDA, **nondeterminism adds no power to TMs.** A deterministic TM can simply
*try every midpoint in turn* &mdash; it has the tape to keep track. The cost is time, not
computability, and that distinction (possible vs. efficient) is where complexity theory
begins.

## 2. Definitions

### Crossing-off, simulated so the algorithm is visible

In [ ]:
def wsw_algorithm(tape, trace=False):
    # the deterministic cross-off algorithm for w#w
    if tape.count('#') != 1: return False
    s = list(tape)
    i = 0
    while i < len(s) and s[i] != '#':
        hashpos = s.index('#')
        j = hashpos + 1 + i
        if j >= len(s) or s[j] != s[i]: return False
        if trace:
            t = s[:]; t[i] = t[i].upper(); t[j] = t[j].upper()
            print("   ", ''.join(t))
        i += 1
    return len(s) == 2 * i + 1

def ww_deterministic(tape):
    # a DTM can TRY every midpoint -- no guessing needed, just more work
    n = len(tape)
    for mid in range(n + 1):
        if tape[:mid] == tape[mid:]:
            return True, mid
    return False, None

### The nondeterministic guess, as a Jove TM

In [ ]:
GuessTM = md2mc('''TM
!! A tiny illustration of GUESSING: at any point the machine may decide
!! "the midpoint is here" and switch to the checking phase.
I : 0 ; 0 , R -> I
I : 1 ; 1 , R -> I
I : 0 ; 0 , R -> C      !! guess: the midpoint is just after this cell
I : 1 ; 1 , R -> C
C : 0 ; 0 , R -> C
C : 1 ; 1 , R -> C
C : . ; . , S -> F
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

## 3. Tests

The $w\#w$ algorithm, crossing off in lock-step.

In [ ]:
print("checking 'abb#abb' :")
ok = wsw_algorithm('abb#abb', trace=True)
print("  ->", ok)
assert ok
print()
print("checking 'abb#aba' :", wsw_algorithm('abb#aba'))
assert not wsw_algorithm('abb#aba')

It is fully deterministic: the `#` removes every choice.

In [ ]:
from itertools import product
cases = [(''.join(a) + '#' + ''.join(b))
         for k in range(3) for a in product('ab', repeat=k)
         for j in range(3) for b in product('ab', repeat=j)]
bad = [c for c in cases
       if wsw_algorithm(c) != (c.split('#')[0] == c.split('#')[1])]
print("mismatches over %d cases :" % len(cases), bad)
assert not bad

**$ww$ has no separator**, so the midpoint must be found.

In [ ]:
for t in ['abab', 'abba', 'aa', 'aba', '']:
    ok, mid = ww_deterministic(t)
    print("  %-7r is ww? %-6s midpoint %s" % (t, ok, mid))
assert ww_deterministic('abab')[0] and not ww_deterministic('abba')[0]

A nondeterministic machine **guesses**; Jove explores all the guesses at once.

In [ ]:
nd = [(k, sorted(v)) for k, v in GuessTM["Delta"].items() if len(v) > 1]
print("nondeterministic entries :", len(nd))
assert nd
print("accepts '0101' ?", tm_accepts(GuessTM, '0101', fuel=80))

**But a DTM can try every midpoint** &mdash; nondeterminism costs time, not power.

In [ ]:
import itertools
def ww_dtm_cost(t):
    tries = 0
    for mid in range(len(t) + 1):
        tries += 1
        if t[:mid] == t[mid:]: return True, tries
    return False, tries

for t in ['abab', 'aabbaabb', 'abcabc']:
    ok, tries = ww_dtm_cost(t)
    print("  %-10r ww? %-6s midpoints tried %d" % (t, ok, tries))
print("\nn+1 attempts instead of one lucky guess.  Slower, not weaker.")
print("(Contrast Chapter 12: a DPDA genuinely CANNOT do the palindrome language.)")

## 4. Animation

The guessing machine: the fork out of `I` is the midpoint guess.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateTM import *
AnimateTM(GuessTM, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write the $w\#w$ machine out as Jove TM markdown. How many states?
2. Why can a DTM simulate an NDTM but a DPDA cannot simulate an NPDA?
3. What is the time cost of the general NDTM-to-DTM simulation?

In [ ]:
# Your work for the exercises above.